# Model selection

## What is polynomial regression?

In the [chapter on regression](2:regression), you saw that a model for linear regression with a single predictor takes the form:

$$y_i = \beta_0 + \beta_1 x_i + \epsilon_i$$

Where:
  * $y_i$ is an observation of the outcome.
  * $x_i$ is an observation of the predictor.
  * $\beta_0$ is the intercept.
  * $\beta_1$ is the slope.
  * $\epsilon_i$ is the error.

That model is linear in the parameters, which allows us to derive closed-form solutions for the best parameters (this is ordinary least squares). It is also linear in the predictors, so we assume that the relationship between predictor and outcome is linear. But what if that is not the case? Then you can still use linear regression and ordinary least squares, but you need to transform the predictors (and maybe even the outcome). You can use a [log transformation](2:data_wrangling:log_transformation) for that, but another option is to use polynomial regression.

Polynomial regression models the relationship between the predictor(s) and the outcomes as a polynomial of the predictor(s). Simple linear regression (like the model above) corresponds to a polynomial of degree 1. With a polynomial of degree 2 we get a quadratic model:

$$y_i = \beta_0 + \beta_1 x_i + \beta_2 x_i^2 + \epsilon_i$$

With a polynomial of degree 3 we get a cubic model:

$$y_i = \beta_0 + \beta_1 x_i + \beta_2 x_i^2 + \beta_3 x_i^3 + \epsilon_i$$

With a polynomial of degree 4 we get a quartic model:

$$y_i = \beta_0 + \beta_1 x_i + \beta_2 x_i^2 + \beta_3 x_i^3 + \beta_4 x_i^4 + \epsilon_i$$

With a polynomial of degree 5 we get a quintic model:

$$y_i = \beta_0 + \beta_1 x_i + \beta_2 x_i^2 + \beta_3 x_i^3 + \beta_4 x_i^4 + \beta_5 x_i^5 + \epsilon_i$$

Etc. Adding a new degree adds a new parameter, increasing the flexibility of the model and its ability to capture complex non-linear relationships. But this remains a [multiple linear regression](2:regression:multiple): the model is linear in the parameters, and non-linear in the predictors. This means that you can use [ordinary least squares](2:regression:ols) to find the best parameter values.

```{admonition} Tip: Linear vs non-linear model
:class: tip

Whether a model is linear or non-linear is a matter of reference frame:
  * Linear regression is linear in the parameters and linear in the predictors.
  * Polynomial regression is linear in the parameters and non-linear in the predictors.
  * Logistic regression is non-linear in the parameters and non-linear in the predictors, but it separates the categories linearly.
```

When mutliple predictors are available, polynomial regression can also include the interactions between predictors. For instance, for a cubic model and two predictors $x_1$ and $x_2$:

$$y_i = \beta_0 + \beta_1 x_{1,i} + \beta_2 x_{2,i} + \beta_2 x_{1,i}^2 + \beta_2 x_{2,i}^2 + \beta_3 x_{1,i}^3 + \beta_3 x_{2,i}^3 + \beta_1 x_{1,i} x_{2,i} + \beta_1 x_{1,i}^2 x_{2,i} + \beta_1 x_{1,i} x_{2,i}^2 + \epsilon_i$$

Note that here we only add the interactions whose total degree is at most 3.

## Activity: Sea level vs. CO<sub>2</sub>

Now let's have a look at the impact of the degree of the polynomial on predictive performance. For that, make sure to start the interactive Python environment by clicking on {fa}`rocket` {fa}`arrow-right-long` {guilabel}`Live Code` at the top of this page (then wait until the Python interaction is ready).

First, you need to import some packages:

In [ ]:
from pathlib import Path
import numpy as np
from sklearn.model_selection import KFold

Then, load the data into two NumPy arrays (data from [10.1073/pnas.1216073110](https://doi.org/10.1073/pnas.1216073110)):

In [ ]:
co2 = np.loadtxt(Path.cwd().parent/'../data/sea_level_climate_forcing_foster_rohling_2013.csv', delimiter=',', skiprows=1, usecols=0)
sea_level = np.loadtxt(Path.cwd().parent/'../data/sea_level_climate_forcing_foster_rohling_2013.csv', delimiter=',', skiprows=1, usecols=1)

`co2` contains the CO<sub>2</sub> concentration (in ppmv), the predictor, and `sea_level` contains the relative sea level (in m), the outcome.

The relationship between CO<sub>2</sub> and relative sea level is non-linear, so simple linear regression will not be enough and you need to turn to polynomial regression. To make it easier, let's define a function to create the right design matrix:

In [ ]:
def create_design_matrix(x, degree):
    """
    Creates a design matrix from an array `x` to perform polynomial regression
    of degree `degree`.
    """
    X = np.ones((len(x), degree + 1))
    for i in range(1, degree + 1):
        X[:, i] = x**i

    return X

To quantify the performance of a polynomial model, you can use $k$-fold cross-validation. Compared to the [activity in the previous section](2:validation:activity), you need an extra step: high degrees can lead to really high values and numerical instability, so standardizing the predictors lead to a more robust model.

Standardization in cross-validation is a bit tricky because you need to avoid any data leak between training and test sets. Here's the code snippet to correctly do it when predicting relative sea level from CO<sub>2</sub>:

In [ ]:
degree = 5

# Instantiate the k-fold cross-validator
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Loop over the 5 folds
for i, (train_index, test_index) in enumerate(cv.split(co2)):

    # Get the design matrix for the training set
    X_train = create_design_matrix(co2[train_index], degree)
    y_train = sea_level[train_index]
    # Standardize the predictors in the design matrix
    mean_X_train = X_train[:, 1:].mean(axis=0)
    std_X_train = X_train[:, 1:].std(axis=0)
    X_train[:, 1:] = (X_train[:, 1:] - mean_X_train)/std_X_train

    # Get the design matrix for the test set
    X_test = create_design_matrix(co2[test_index], degree)
    y_test = sea_level[test_index]
    # Standardize the predictors in the design matrix using the mean and standard
    # deviation of the training set
    X_test[:, 1:] = (X_test[:, 1:] - mean_X_train)/std_X_train

Starting from the code snippet above, quantify the performance of a polynomial model of degree 5 using $k$-fold cross-validation with 5 folds and the coefficient of determination as metric. For each fold, you need to:
  * Estimate the best parameters from the training set.
  * Compute the coefficient of determination for the training set.
  * Compute the coefficient of determination for the test set.

That will give you 5 coefficients of determination for the training data, and 5 for the test data. Compute their means to get the final values.

In [ ]:
# Your answer here

Repeat the operation with a polynomial model of degree 2, then one of degree 8. What do you observe? How do the coefficients computed on the training and test sets vary as the degree increases?

In [ ]:
# Your answer here

## What is the bias-variance tradeoff?

### The bias



### The variance



### The tradeoff



## How to select the right model?



### All-subsets variable selection



### Stepwise variable selection

